In [1]:
import re
import nltk
import spacy
import pandas as pd
import numpy as np
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.model_selection import train_test_split
import joblib
from nltk.corpus import stopwords

stop_words = set(stopwords.words("english"))
print("Preprocessing ready!")

Preprocessing ready!


In [2]:
data = pd.read_csv(r"C:\Users\akash\OneDrive\Documents\nlp project\Sentiment Analysis project using NLP + RNN + LSTM,\data\raw\McDonald_s_Reviews.csv",encoding="latin1")
print(data.head(2))

   reviewer_id  store_name              category  \
0            1  McDonald's  Fast food restaurant   
1            2  McDonald's  Fast food restaurant   

                                       store_address  latitude   longitude  \
0  13749 US-183 Hwy, Austin, TX 78750, United States  30.460718 -97.792874   
1  13749 US-183 Hwy, Austin, TX 78750, United States  30.460718 -97.792874   

  rating_count   review_time  \
0        1,240  3 months ago   
1        1,240    5 days ago   

                                              review   rating  
0  Why does it look like someone spit on my food?...   1 star  
1  It'd McDonalds. It is what it is as far as the...  4 stars  


In [3]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+|www\S+", "", text)          # remove links
    text = re.sub(r"@\w+", "", text)                    # remove @mentions
    text = re.sub(r"#\w+", "", text)                    # remove hashtags
    text = re.sub(r"[^a-zA-Z\s]", " ", text)            # keep letters only
    text = re.sub(r"\s+", " ", text).strip()            # remove extra spaces
    return text
data["clean_review"]=data["review"].apply(clean_text)

In [4]:
data.columns

Index(['reviewer_id', 'store_name', 'category', 'store_address', 'latitude ',
       'longitude', 'rating_count', 'review_time', 'review', 'rating',
       'clean_review'],
      dtype='str')

In [5]:
data = data.drop(
    columns=['reviewer_id', 'store_name', 'category', 'store_address', 'latitude ',
       'longitude', 'rating_count', 'review_time', 
       ]
)

In [6]:
data

,review,rating,clean_review
0,Why does it look like someone spit on my food?...,1 star,why does it look like someone spit on my food ...
1,It'd McDonalds. It is what it is as far as the...,4 stars,it d mcdonalds it is what it is as far as the ...
2,Made a mobile order got to the speaker and che...,1 star,made a mobile order got to the speaker and che...
3,My mc. Crispy chicken sandwich was ï¿½ï¿½ï¿½ï¿...,5 stars,my mc crispy chicken sandwich was customer ser...
4,"I repeat my order 3 times in the drive thru, a...",1 star,i repeat my order times in the drive thru and ...
...,...,...,...
33391,They treated me very badly.,1 star,they treated me very badly
33392,The service is very good,5 stars,the service is very good
33393,To remove hunger is enough,4 stars,to remove hunger is enough
33394,"It's good, but lately it has become very expen...",5 stars,it s good but lately it has become very expensive


In [7]:
data["rating"].value_counts()

rating
5 stars    10274
1 star      9431
4 stars     5787
3 stars     4818
2 stars     3086
Name: count, dtype: int64

In [8]:
data["sentiment"] = data["rating"].replace({
    "5 stars": 1,
    "4 stars": 1,
    "3 stars": 2,
    "2 stars": 0,
    "1 star": 0
})

In [9]:
data=data.drop(columns=["rating"])

In [10]:
data["sentiment"].info()


<class 'pandas.Series'>
RangeIndex: 33396 entries, 0 to 33395
Series name: sentiment
Non-Null Count  Dtype 
--------------  ----- 
33396 non-null  object
dtypes: object(1)
memory usage: 261.0+ KB


In [11]:
x=data["clean_review"]
y=data["sentiment"]

In [12]:
X_train, X_test, y_train, y_test = train_test_split(
    x, y, test_size=0.2, random_state=42
)


In [13]:
tokenizer = Tokenizer(num_words=10000)
tokenizer.fit_on_texts(X_train)

In [14]:
print(len(tokenizer.word_index))

12338


In [15]:
X_train_seq = tokenizer.texts_to_sequences(X_train) 
X_test_seq = tokenizer.texts_to_sequences(X_test) 

In [16]:
maxlimt = 300
X_train_ped = pad_sequences(X_train_seq,maxlen=maxlimt,padding="post")
X_test_ped = pad_sequences(X_test_seq,maxlen=maxlimt,padding="post")

In [17]:
joblib.dump(tokenizer, "tokenizer.pkl")

['tokenizer.pkl']

In [18]:
clean_data =pd.concat([x,y],axis=1)

In [19]:
clean_data.to_csv(r"C:\Users\akash\OneDrive\Documents\nlp project\Sentiment Analysis project using NLP + RNN + LSTM,\data\clean_data\clean_data.csv")

In [20]:
joblib.dump([X_train_ped,X_test_ped,y_train, y_test],r"C:\Users\akash\OneDrive\Documents\nlp project\Sentiment Analysis project using NLP + RNN + LSTM,\model\split.pkl")

['C:\\Users\\akash\\OneDrive\\Documents\\nlp project\\Sentiment Analysis project using NLP + RNN + LSTM,\\model\\split.pkl']

In [21]:
data["sentiment"].value_counts()

sentiment
1    16061
0    12517
2     4818
Name: count, dtype: int64